# 03 · CPS transitions and outcomes

Part 3 of 5. Builds the observed occupation-to-occupation transition matrix from CPS ASEC 2015–2025, tests skill similarity against it, and measures how much of it is coding noise.

**Reads:** `data/raw/cps_00001.csv`, `data/derived/job-features.parquet`, `data/derived/census-onet-crosswalk.parquet`
**Writes:** `data/derived/census-transitions.parquet`, `baseline-results.csv`; `data/interim/census-similarity.parquet` (large, regenerated), `data/interim/cps-workers.parquet` and `data/interim/linked-movers.parquet` (person-level, gitignored: IPUMS terms forbid redistributing individual rows)

**Decisions**
- Workers: employed now (EMPSTAT 10/12), 26+ weeks worked last year, both occupations mappable. A mover is anyone with OCC10LY ≠ OCC2010.
- Flows are weighted by ASECWT.

**Results**
- 721,769 workers, 80,032 movers (11.1%).
- Skill similarity puts the real destination in the top 10 for 25.2% of movers, against 15.5% for popularity and 1.9% random.
- Robust to coding noise: 23.6% on movers who also changed industry. Industry codes are reliable (9.6% of stayers change industry vs 87% of movers).
- Linked across two ASEC years, 32% of reported moves are confirmed against a 49% ceiling for stayers, so roughly two-thirds of reported moves are real.
- Confirmed movers' median log wage change is 0.095 vs 0.041 for stayers.

In [1]:
import numpy as np
import pandas as pd

cps = pd.read_csv("../data/raw/cps_00001.csv")
feat = pd.read_parquet("../data/derived/job-features.parquet").set_index("O*NET-SOC Code")
census_onet = pd.read_parquet("../data/derived/census-onet-crosswalk.parquet")
print(cps.shape, feat.shape, census_onet.shape)

(1083340, 26) (910, 110) (940, 2)


## Census-level similarity and the transition matrix

In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# 1) collapse O*NET features to census level (mean over all O*NET codes under each census code)
feat_c = (census_onet.merge(feat, left_on='onet', right_index=True)
          .drop(columns='onet').groupby('census').mean())
Xc = StandardScaler().fit_transform(feat_c)
Sc = pd.DataFrame(cosine_similarity(Xc), index=feat_c.index, columns=feat_c.index)
print(Sc.shape, 'census-level similarity')

# 2) observed transitions: employed now, 26+ weeks last year, both occupations mappable
m = cps[cps['EMPSTAT'].isin([10, 12])
        & (cps['WKSWORK1'] >= 26)
        & cps['OCC2010'].isin(feat_c.index)
        & cps['OCC10LY'].isin(feat_c.index)]
movers = m[m['OCC2010'] != m['OCC10LY']]
print(len(m), 'workers;', len(movers), f'movers ({len(movers)/len(m):.1%})')

# weighted origin x destination matrix, diagonal excluded by construction
T = movers.pivot_table(index='OCC10LY', columns='OCC2010', values='ASECWT', aggfunc='sum', fill_value=0)
print(T.shape, 'transition matrix')

(517, 517) census-level similarity
721769 workers; 80032 movers (11.1%)


(459, 459) transition matrix


## Kill question: does skill similarity predict real moves?

hit@10 is the weighted share of movers whose destination is in the top 10; MRR is the mean of 1/rank.

In [3]:
import numpy as np

dests = Sc.columns
origins = [o for o in T.index if o in Sc.index and T.loc[o].sum() > 0]

# popularity baseline: total mover inflow per destination, across all origins
pop = movers.groupby('OCC2010')['ASECWT'].sum().reindex(dests, fill_value=0)

def evaluate(score_fn, k=10):
    hits, rr, total = 0.0, 0.0, 0.0
    for o in origins:
        flows = T.loc[o].reindex(dests, fill_value=0).drop(o)
        ranks = score_fn(o).drop(o).rank(ascending=False, method='first')
        hits += flows[ranks <= k].sum()
        rr += (flows / ranks).sum()
        total += flows.sum()
    return hits / total, rr / total

results = pd.DataFrame(
    [evaluate(lambda o: Sc.loc[o]),
     evaluate(lambda o: pop),
     (10 / (len(dests) - 1), np.nan)],
    index=['skill similarity', 'popularity', 'random'],
    columns=['hit@10', 'MRR'])
print(results.round(3))

                  hit@10    MRR
skill similarity   0.252  0.119
popularity         0.155  0.071
random             0.019    NaN


In [4]:
Sc.reset_index().to_parquet('../data/interim/census-similarity.parquet', index=False)
T.reset_index().to_parquet('../data/derived/census-transitions.parquet', index=False)
results.to_csv('../data/derived/baseline-results.csv')

## Robustness: movers who also changed industry

In [5]:
def evaluate(T, mv, score_fn, k=10):
    pop = mv.groupby('OCC2010')['ASECWT'].sum().reindex(dests, fill_value=0)
    origins = [o for o in T.index if o in Sc.index and T.loc[o].sum() > 0]
    hits, rr, total = 0.0, 0.0, 0.0
    for o in origins:
        flows = T.loc[o].reindex(dests, fill_value=0).drop(o)
        ranks = score_fn(o, pop).drop(o).rank(ascending=False, method='first')
        hits += flows[ranks <= k].sum()
        rr += (flows / ranks).sum()
        total += flows.sum()
    return hits / total, rr / total

def run(mv, label):
    T = mv.pivot_table(index='OCC10LY', columns='OCC2010', values='ASECWT', aggfunc='sum', fill_value=0)
    sim = evaluate(T, mv, lambda o, pop: Sc.loc[o])
    popl = evaluate(T, mv, lambda o, pop: pop)
    return pd.DataFrame([sim, popl], index=[f'{label}: similarity', f'{label}: popularity'],
                        columns=['hit@10', 'MRR'])

ind_changers = movers[movers['IND90LY'] != movers['IND1990']]
print(len(ind_changers), f'industry changers ({len(ind_changers)/len(movers):.0%} of movers)')

print(pd.concat([run(movers, 'all movers'), run(ind_changers, 'industry changers')]).round(3))


69574 industry changers (87% of movers)


                               hit@10    MRR
all movers: similarity          0.252  0.119
all movers: popularity          0.155  0.071
industry changers: similarity   0.236  0.107
industry changers: popularity   0.162  0.074


## Are industry codes reliable? Stayers as a noise floor

In [6]:
stayers = m[m['OCC2010'] == m['OCC10LY']]

def ind_change_rate(df):
    changed = df['IND90LY'] != df['IND1990']
    return (df.loc[changed, 'ASECWT'].sum() / df['ASECWT'].sum())

print(f"stayers: {ind_change_rate(stayers):.1%} changed industry  (n={len(stayers):,})")
print(f"movers:  {ind_change_rate(movers):.1%} changed industry  (n={len(movers):,})")

stayers: 9.6% changed industry  (n=641,737)
movers:  87.0% changed industry  (n=80,032)


## Linking people across two ASEC years

First pass: raw year-over-year switches (41%) are mostly coding noise. Second pass requires a move reported in year 0 to be confirmed by an independent coding in year 1.

In [7]:
# people appearing in consecutive ASEC years, linked on the validated person ID
a = cps[['CPSIDV', 'YEAR', 'OCC10LY', 'INCWAGE', 'AGE', 'SEX']]
linked = a.merge(a, on='CPSIDV', suffixes=('_t0', '_t1'))
linked = linked[linked['YEAR_t1'] == linked['YEAR_t0'] + 1]

# sanity: same person should age ~1 year and keep sex
ok = ((linked['AGE_t1'] - linked['AGE_t0']).between(0, 2)) & (linked['SEX_t1'] == linked['SEX_t0'])
linked = linked[ok]

switch = linked[(linked['OCC10LY_t0'] != linked['OCC10LY_t1'])
                & ~linked['OCC10LY_t0'].isin([0, 9999]) & ~linked['OCC10LY_t1'].isin([0, 9999])]
print(len(linked), 'linked person-pairs;', len(switch), 'with an occupation change between years')

290507 linked person-pairs; 119740 with an occupation change between years


In [8]:
cols = ['CPSIDV', 'YEAR', 'AGE', 'SEX', 'OCC10LY', 'OCC2010', 'INCWAGE', 'WKSWORK1']
a = cps[cols]
linked = a.merge(a, on='CPSIDV', suffixes=('_t0', '_t1'))
linked = linked[linked['YEAR_t1'] == linked['YEAR_t0'] + 1]

# same person: ages ~1 year, same sex
linked = linked[(linked['AGE_t1'] - linked['AGE_t0']).between(0, 2)
                & (linked['SEX_t1'] == linked['SEX_t0'])]

# substantial work and positive wages in both years
linked = linked[(linked['WKSWORK1_t0'] >= 26) & (linked['WKSWORK1_t1'] >= 26)
                & (linked['INCWAGE_t0'] > 0) & (linked['INCWAGE_t1'] > 0)]

valid = ~linked['OCC10LY_t0'].isin([0, 9999]) & ~linked['OCC2010_t0'].isin([0, 9999])
reported_move = valid & (linked['OCC10LY_t0'] != linked['OCC2010_t0'])
confirmed = reported_move & (linked['OCC10LY_t1'] == linked['OCC2010_t0'])
stayed = valid & (linked['OCC10LY_t0'] == linked['OCC2010_t0']) & (linked['OCC10LY_t1'] == linked['OCC2010_t0'])

print(f"{len(linked):,} linked pairs after filters")
print(f"{reported_move.sum():,} reported a move in year 0")
print(f"{confirmed.sum():,} confirmed by year 1 ({confirmed.sum()/reported_move.sum():.0%} of reported)")
print(f"{stayed.sum():,} confirmed stayers (comparison group)")

moves = linked[confirmed].copy()
moves['log_wage_change'] = np.log(moves['INCWAGE_t1']) - np.log(moves['INCWAGE_t0'])
stays = linked[stayed].copy()
stays['log_wage_change'] = np.log(stays['INCWAGE_t1']) - np.log(stays['INCWAGE_t0'])
print(f"median log wage change: movers {moves['log_wage_change'].median():.3f}, "
      f"stayers {stays['log_wage_change'].median():.3f}")

182,636 linked pairs after filters
18,449 reported a move in year 0
5,818 confirmed by year 1 (32% of reported)
79,522 confirmed stayers (comparison group)
median log wage change: movers 0.095, stayers 0.041


### Calibrating confirmation against stayers

In [9]:
reported_stay = valid & (linked['OCC10LY_t0'] == linked['OCC2010_t0'])
print(f"stayers confirmed: {stayed.sum()/reported_stay.sum():.0%}")
print(f"movers confirmed:  {confirmed.sum()/reported_move.sum():.0%}")

stayers confirmed: 49%
movers confirmed:  32%


## Save hand-off files

In [10]:
m.to_parquet('../data/interim/cps-workers.parquet')
pd.concat([moves.assign(group='confirmed mover'), stays.assign(group='confirmed stayer')]) \
    .to_parquet('../data/interim/linked-movers.parquet')
print(m.shape, len(moves), len(stays), 'saved')

(721769, 26) 5818 79522 saved
